# 2.1 — Cadastro refinado de materiais

- **Propósito:** Sanitizar e consolidar o cadastro de materiais recebido do SAP.
- **Entrada:** `pr_cadastrao/sap_cadastraorefinado/current`
- **Saídas:** `pr_cadastrao.material_cadastrao` e `_agents_databases.material_cadastrao`
- **Chave:** Empresa + Material · **Carga:** Completa

In [0]:
import re
import unicodedata

# Função para sanitizar nomes de colunas: remove acentos, converte para lowercase e substitui caracteres inválidos por underscore
def sanitize_col_name(name):
    if not name:
        return "col_unknown"
    # Remover acentos
    nfkd = unicodedata.normalize('NFKD', name)
    ascii_name = nfkd.encode('ASCII', 'ignore').decode('ASCII')
    # Substituir caracteres inválidos por underscore
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', ascii_name)
    # Remover underscores consecutivos e nas pontas
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean.lower()

In [0]:
# Leitura dos arquivos Excel - todas as colunas como STRING para preservar zeros à esquerda
df_raw = (spark.read
    .format("excel")
    .option("header", "true")
    .option("inferSchema", "false")
    .load("/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/")
)

print(f"Arquivos carregados: {len(df_raw.columns)} colunas, {df_raw.count()} linhas (incluindo possíveis headers duplicados)")

In [0]:
# Verificar se o header foi aplicado corretamente pelo reader
first_col = df_raw.columns[0]

if first_col.startswith("_c"):
    # Header não foi aplicado - extrair nomes da primeira linha e renomear
    header_row = df_raw.first()
    original_names = [header_row[i] if header_row[i] else f"col_{i}" for i in range(len(df_raw.columns))]
    sanitized_names = [sanitize_col_name(n) for n in original_names]

    # Tratar duplicatas adicionando sufixo incremental
    seen = {}
    for i, name in enumerate(sanitized_names):
        if name in seen:
            seen[name] += 1
            sanitized_names[i] = f"{name}_{seen[name]}"
        else:
            seen[name] = 0

    df = df_raw.toDF(*sanitized_names)

    # Remover linhas de cabeçalho de TODOS os arquivos (cada .xlsx carrega seu próprio header)
    header_values = [v for v in [header_row[0], header_row[1], header_row[2]] if v]
    df = df.filter(
        ~(
            (df[sanitized_names[0]] == header_values[0]) &
            (df[sanitized_names[1]] == header_values[1]) &
            (df[sanitized_names[2]] == header_values[2])
        )
    )
else:
    # Header aplicado - apenas sanitizar os nomes existentes
    original_names = df_raw.columns
    sanitized_names = [sanitize_col_name(n) for n in original_names]

    seen = {}
    for i, name in enumerate(sanitized_names):
        if name in seen:
            seen[name] += 1
            sanitized_names[i] = f"{name}_{seen[name]}"
        else:
            seen[name] = 0

    df = df_raw.toDF(*sanitized_names)

print(f"DataFrame final: {df.count()} linhas, {len(df.columns)} colunas")
print(f"Primeiras colunas: {df.columns[:10]}")

In [0]:
from pyspark.sql import functions as F
import uuid

# Configurações de auditoria
CURRENT_USER = spark.sql("SELECT current_user()").first()[0]
LOAD_MODE = "full"
LOAD_ID = str(uuid.uuid4())

# Adicionar colunas de auditoria/metadados
df = df.withColumns({
    "_ingested_at": F.from_utc_timestamp(F.current_timestamp(), "America/Sao_Paulo"),
    "_last_updated_at": F.from_utc_timestamp(F.current_timestamp(), "America/Sao_Paulo"),
    "_ingested_by": F.lit(CURRENT_USER),
    "_load_type": F.lit(LOAD_MODE),
    "_load_id": F.lit(LOAD_ID),
    "_source_file_path": F.lit("/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/"),
})

print(f"Colunas de auditoria adicionadas: _ingested_at, _last_updated_at, _ingested_by, _load_type, _load_id, _source_file_path")
print(f"Load ID: {LOAD_ID}")
print(f"Ingested by: {CURRENT_USER}")

In [0]:
# Dropar tabela antiga se existir
spark.sql("DROP TABLE IF EXISTS parts_hdbk_sandbox.pr_cadastrao.material_cadastrao")

# Criar tabela Delta
df.write.saveAsTable("parts_hdbk_sandbox.pr_cadastrao.material_cadastrao")

print(f"Tabela parts_hdbk_sandbox.pr_cadastrao.material_cadastrao criada com sucesso.")
print(f"Total de registros: {df.count()}")

In [0]:
# Saída adicional: replicar no schema _agents_databases
AGENTS_TABLE = "parts_hdbk_sandbox._agents_databases.material_cadastrao"

spark.sql(f"DROP TABLE IF EXISTS {AGENTS_TABLE}")
df.write.saveAsTable(AGENTS_TABLE)

print(f"Tabela {AGENTS_TABLE} criada com sucesso.")
print(f"Total de registros: {df.count()}")

In [0]:
%sql
-- Definir colunas NOT NULL (requisito para PK)
ALTER TABLE parts_hdbk_sandbox.pr_cadastrao.material_cadastrao
ALTER COLUMN empresa SET NOT NULL;

ALTER TABLE parts_hdbk_sandbox.pr_cadastrao.material_cadastrao
ALTER COLUMN material SET NOT NULL;

-- Adicionar constraint de chave primária composta
ALTER TABLE parts_hdbk_sandbox.pr_cadastrao.material_cadastrao
ADD CONSTRAINT pk_material_cadastrao PRIMARY KEY (empresa, material);

In [0]:
# =============================================================================
# 1. DICIONÁRIO COM COMENTÁRIOS DOS CAMPOS
# =============================================================================
RAW_TABLE = "parts_hdbk_sandbox.pr_cadastrao.material_cadastrao"

COLUMN_COMMENTS = {
    # --- Identificação do Material ---
    "centro": "Código do centro/depósito SAP que atende o material (ex: 0203=Sumaré 2W, 0503=Sumaré 4W)",
    "material": "Código único do material/peça (partnumber SAP). Parte da chave primária.",
    "empresa": "Código da empresa SAP (ex: 0200=2W, 0500=4W). Parte da chave primária.",
    "descricao_curta_portugues": "Descrição curta do material em português",
    "descricao_curta_ingles": "Descrição curta do material em inglês",
    "descricao_curta_espanhol": "Descrição curta do material em espanhol",
    "unidade_de_medida_basica": "Unidade de medida básica do material (ex: PC, KG, M)",
    "numero_material_antigo": "Número do material no sistema legado/antigo",
    "setor_de_atividade": "Código do setor de atividade (ex: 04=Automotivo)",
    "status_material": "Status do material no SAP (ex: N=Normal, R=Restrito, H=Histórico)",

    # --- Características Físicas ---
    "peso_bruto": "Peso bruto do material (com embalagem)",
    "unidade_peso": "Unidade de medida do peso (ex: KG)",
    "peso_liquido": "Peso líquido do material (sem embalagem)",
    "tipo_de_material": "Tipo de material SAP (ex: ZRO1, ZHAW, ZFER)",
    "codigo_cor": "Código da cor do material",
    "descricao_cor": "Descrição da cor do material",

    # --- Cadeia e Intercambiabilidade ---
    "intercambiabilidade": "Indica se o material possui intercambiabilidade com outros",
    "item_principal_cadeia": "Material principal na cadeia de substituição",
    "data_cadeia": "Data de início da cadeia de substituição",
    "cadeia": "Código da cadeia de substituição",

    # --- Segmentação Comercial ---
    "modelo_comercial_principal": "Modelo comercial principal ao qual o material está associado",
    "segmento_principal": "Segmento de negócio principal (ex: 4W, 2W)",
    "sub_segmento_principal": "Sub-segmento de negócio (ex: PAS=Passenger, COM=Commercial)",
    "cut_in_material": "Data de início de utilização do material (cut-in)",
    "cut_off_material": "Data de fim de utilização do material (cut-off)",
    "ciclo_de_vida": "Código do ciclo de vida do material (ex: Z01, Z05)",

    # --- Vendas e Margem ---
    "status_venda_mi": "Status de venda no mercado interno",
    "status_venda_me": "Status de venda no mercado externo (exportação)",
    "codigo_de_margem": "Código de classificação de margem do material",
    "origem": "Código de origem do material (1=Nacional, 3=Importado)",

    # --- Comercial e Fiscal ---
    "embalagem_comercial": "Indicador de embalagem comercial",
    "codigo_de_ncm": "Código NCM (Nomenclatura Comum do Mercosul) para classificação fiscal",
    "pais_de_origem": "País de origem do material (ex: BR, JP)",
    "pis": "Alíquota de PIS aplicada ao material",
    "cofins": "Alíquota de COFINS aplicada ao material",
    "ipi_na_venda_price_por_ncm": "Alíquota de IPI na venda, determinada pelo NCM",

    # --- Suprimentos e MRP ---
    "grupo_de_compradores": "Código do grupo de compradores responsável",
    "material_codigo_de_fabrica": "Código do material na fábrica (com sufixo de cor/variante)",
    "tipo_de_mrp": "Tipo de planejamento MRP (ex: PD=Planejamento determinístico, V1=Manual)",
    "lote_de_compra": "Tamanho do lote mínimo de compra",
    "estoque_reserva": "Quantidade de estoque de reserva/segurança",
    "tipo_de_suprimento": "Tipo de suprimento (ex: X=Compra externa, F=Fabricação própria)",
    "suprimento_especial": "Código de suprimento especial (ex: 20=Consignação)",
    "prazo_entrega_prevista_lead_time_firme": "Lead time firme de entrega em dias",
    "fornecedor": "Código do fornecedor principal no SAP",
    "descricao_do_fornecedor": "Razão social do fornecedor principal",
    "codigo_de_reabastecimento": "Código que define a regra de reabastecimento",
    "lead_time_no_scm": "Lead time utilizado no planejamento SCM",
    "status_compra": "Status de compra do material",

    # --- Estoques ---
    "estoque_livre_no_centro": "Quantidade de estoque livre (disponível) no centro",
    "estoque_disponivel_para_venda": "Quantidade de estoque disponível para venda",
    "estoque_bloqueado": "Quantidade de estoque bloqueado (indisponível)",
    "estoque_em_transito": "Quantidade de estoque em trânsito entre centros",
    "estoque_em_poder_de_terceiros": "Quantidade de estoque em poder de terceiros",
    "estoque_em_controle_qualidade": "Quantidade de estoque em inspeção de qualidade",
    "estoque_devolucoes": "Quantidade de estoque em devoluções",
    "estoque_de_seguranca": "Nível de estoque de segurança definido no MRP",
    "valor_do_estoque": "Valor monetário total do estoque no centro",

    # --- Pedidos e Produção ---
    "saldo_da_carteira_de_pedidos": "Saldo pendente na carteira de pedidos de clientes",
    "ordem_de_producao": "Quantidade em ordens de produção abertas",
    "quantidade_em_pi": "Quantidade em Pedidos de Importação (PI)",
    "quantidade_em_bo": "Quantidade em Back Order (BO) - pedidos pendentes",

    # --- Preços ---
    "preco_de_rede_price_de_venda_liquida": "Preço de venda líquida (rede)",
    "preco_de_rede_price_de_venda_bruta_base_sp": "Preço de venda bruta base São Paulo",
    "preco_de_exportacao_price_de_venda": "Preço de venda para exportação",
    "moeda_do_preco_liquido_info_record": "Moeda do preço líquido no Info Record SAP",
    "preco_liquido_info_record": "Preço líquido registrado no Info Record de compras",
    "centro_de_lucro": "Código do centro de lucro associado ao material",

    # --- Atributos HUB ---
    "modelo_comercial_principal_hub": "Modelo comercial principal no contexto HUB",
    "segmento_principal_hub": "Segmento principal no contexto HUB",
    "sub_segmento_principal_hub": "Sub-segmento principal no contexto HUB",
    "cut_in_material_hub": "Data de cut-in no contexto HUB",
    "cut_off_material_hub": "Data de cut-off no contexto HUB",

    # --- Família de Produto ---
    "familia_nivel_1": "Classificação de família do produto - Nível 1 (mais agregado)",
    "familia_nivel_2": "Classificação de família do produto - Nível 2",
    "familia_nivel_3": "Classificação de família do produto - Nível 3 (mais detalhado)",

    # --- Outros ---
    "atributo_produto_1": "Atributo adicional do produto (campo 1)",
    "atributo_produto_2": "Atributo adicional do produto (campo 2)",
    "function_code": "Código de função do material",
    "sub_grupo": "Sub-grupo de classificação do material",
    "periodo_qip": "Período QIP (Quality Inspection Period)",
    "grp_mat_2": "Grupo de material secundário",

    # --- Auditoria ---
    "_ingested_at": "Data e hora da ingestão dos dados (timezone America/Sao_Paulo)",
    "_last_updated_at": "Data e hora da última atualização dos dados",
    "_ingested_by": "Usuário responsável pela ingestão dos dados",
    "_load_type": "Tipo de carga (full=carga completa, incremental=incremental)",
    "_load_id": "Identificador único da execução de carga (UUID)",
    "_source_file_path": "Caminho do diretório fonte dos arquivos ingeridos",
}

# =============================================================================
# 2. COMENTÁRIO DA TABELA
# =============================================================================
TABLE_COMMENT = """
Camada Raw do Cadastro de Materiais SAP. Dados preservados como STRING sem conversao de tipo.

Chave Primaria: empresa + material
Atualizacao: Carga completa (full overwrite)
Fonte: /Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/

Relacionamentos:
  - material -> tabela de produtos/SKUs (raw_sales_order)
  - empresa -> segmento de negocio (0200=2W, 0500=4W)
  - centro -> deposito de atendimento
  - fornecedor -> tabela de fornecedores
"""

spark.sql(f"""
    COMMENT ON TABLE {RAW_TABLE} IS '{TABLE_COMMENT.replace(chr(39), chr(39)+chr(39))}'
""")

# =============================================================================
# 3. COMENTÁRIOS NAS COLUNAS
# =============================================================================
for column_name, comment in COLUMN_COMMENTS.items():
    escaped_comment = comment.replace("'", "''")
    try:
        spark.sql(f"COMMENT ON COLUMN {RAW_TABLE}.{column_name} IS '{escaped_comment}'")
    except Exception as e:
        print(f"  [WARN] Coluna '{column_name}' não encontrada: {e}")

print(f"Comentarios adicionados para {len(COLUMN_COMMENTS)} colunas")

# =============================================================================
# 4. TAGS DO UNITY CATALOG
# =============================================================================
spark.sql(f"""
    ALTER TABLE {RAW_TABLE} SET TAGS (
        'domain' = 'materials',
        'layer' = 'raw',
        'source' = 'sap',
        'data_classification' = 'internal'
    )
""")

spark.sql(f"ALTER TABLE {RAW_TABLE} ALTER COLUMN empresa SET TAGS ('business_key' = 'true')")
spark.sql(f"ALTER TABLE {RAW_TABLE} ALTER COLUMN material SET TAGS ('business_key' = 'true', 'joins_to' = 'raw_sales_order')")
spark.sql(f"ALTER TABLE {RAW_TABLE} ALTER COLUMN centro SET TAGS ('business_key' = 'true')")
spark.sql(f"ALTER TABLE {RAW_TABLE} ALTER COLUMN codigo_de_ncm SET TAGS ('fiscal' = 'true')")

print("Tags aplicadas na tabela e colunas")

# =============================================================================
# 5. PROPRIEDADES CUSTOMIZADAS (TBLPROPERTIES)
# =============================================================================
spark.sql(f"""
    ALTER TABLE {RAW_TABLE} SET TBLPROPERTIES (
        'business_owner' = 'Demand Planning',
        'technical_owner' = 'Andre Causs',
        'data_domain' = 'Materials Master Data',
        'source_system' = 'SAP',
        'refresh_frequency' = 'full_load',
        'primary_key' = 'empresa, material'
    )
""")

print("Propriedades customizadas configuradas")
print(f"\nMetadados completos aplicados a tabela {RAW_TABLE}")

# =============================================================================
# 6. METADADOS DA CÓPIA _agents_databases
# =============================================================================
AGENTS_TABLE = "parts_hdbk_sandbox._agents_databases.material_cadastrao"
AGENTS_COMMENT = """
Cópia do Cadastro de Materiais SAP para agentes AI. Dados preservados como STRING.

Modelo: Overwrite completo (espelho de pr_cadastrao.material_cadastrao)
Chave Primária: empresa + material
Fonte: /Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/
"""

try:
    spark.sql(
        f"COMMENT ON TABLE {AGENTS_TABLE} IS "
        f"'{AGENTS_COMMENT.replace(chr(39), chr(39)+chr(39))}'"
    )
    # Aplicar comentários nas colunas-chave
    AGENTS_KEY_COLS = {
        "empresa": "Código da empresa SAP (ex: 0200=2W, 0500=4W). Parte da chave primária.",
        "material": "Código único do material/peça (partnumber SAP). Parte da chave primária.",
        "centro": "Código do centro/depósito SAP.",
        "descricao_curta_portugues": "Descrição curta do material em português.",
        "tipo_de_material": "Tipo de material SAP (ex: ZRO1, ZHAW, ZFER).",
        "item_principal_cadeia": "Material principal na cadeia de substituição.",
        "modelo_comercial_principal": "Modelo comercial principal associado ao material.",
        "segmento_principal": "Segmento de negócio principal (ex: 4W, 2W).",
        "status_material": "Status do material no SAP.",
        "fornecedor": "Código do fornecedor principal no SAP.",
        "descricao_do_fornecedor": "Razão social do fornecedor principal.",
        "_ingested_at": "Data e hora da ingestão dos dados.",
        "_ingested_by": "Usuário responsável pela ingestão.",
        "_load_type": "Tipo de carga (full=carga completa).",
        "_load_id": "Identificador único da execução de carga (UUID).",
    }
    for col_name, comment in AGENTS_KEY_COLS.items():
        escaped = comment.replace("'", "''")
        spark.sql(f"COMMENT ON COLUMN {AGENTS_TABLE}.`{col_name}` IS '{escaped}'")

    spark.sql(f"""
        ALTER TABLE {AGENTS_TABLE} SET TAGS (
            'domain' = 'materials', 'layer' = 'raw',
            'source' = 'sap', 'history_model' = 'full_overwrite',
            'data_classification' = 'internal'
        )
    """)
    spark.sql(f"""
        ALTER TABLE {AGENTS_TABLE} SET TBLPROPERTIES (
            'business_owner' = 'Demand Planning',
            'technical_owner' = 'Andre Causs',
            'data_domain' = 'Materials Master Data',
            'source_system' = 'SAP',
            'refresh_frequency' = 'full_overwrite',
            'natural_key' = 'empresa, material'
        )
    """)
    print(f"\nMetadados aplicados a tabela {AGENTS_TABLE}")
except Exception as e:
    print(f"\n[WARN] Falha ao aplicar metadados em {AGENTS_TABLE}: {e}")